# FairGAD — Reddit Benchmark

**Kernel:** set to `.venv` (Python 3.8).  
If the kernel isn't registered yet, run this once in a terminal first:
```bash
cd "/Users/ashrafshahreyar/Coding/Data Mining/FairGAD"
source .venv/bin/activate
.venv/bin/python -m pip install ipykernel
.venv/bin/python -m ipykernel install --user --name fairgad --display-name "Python (fairgad)"
```
Then select **Python (fairgad)** as the kernel.

In [1]:
import sys, gc, warnings
from pathlib import Path

import numpy as np
import torch
import tqdm
import pandas as pd

from torch_geometric.seed import seed_everything
from pygod.metrics import eval_roc_auc, statistical_parity, equality_of_odds
from pygod.models import CONAD, CoLA, DOMINANT, AdONE, DONE, ONE_NEW, VGOD
from sklearn.metrics import precision_recall_curve, auc

warnings.filterwarnings('ignore')
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| MPS:', torch.backends.mps.is_available())

torch: 2.1.0 | CUDA: False | MPS: True


In [2]:
# ── Configuration (matches paper / config.json) ────────────────────────────
DATA_ROOT   = Path('/Users/ashrafshahreyar/Coding/Data Mining/FairGAD/data')
DATASET     = 'our_reddit'
NUM_TRIALS  = 20
SEED        = 1
GPU         = -1          # -1 → CPU (no CUDA on this machine)
BATCH_SIZE  = 16384
FAIR_FACTOR = 0.0001      # from config.json
ADCG_FACTOR = 10          # from config.json

MODELS = ['DOMINANT', 'CONAD', 'COLA', 'ADONE', 'DONE', 'ONE', 'VGOD']

# Each entry: (regulariser, fair_factor, adcg_factor, label)
CONDITIONS = [
    ('none',        0,           0,           'Baseline'),
    ('hin',         FAIR_FACTOR, 0,           'HIN'),
    ('fairod',      FAIR_FACTOR, 0,           'FairOD'),
    ('correlation', FAIR_FACTOR, 0,           'Correlation'),
    ('hin',         FAIR_FACTOR, ADCG_FACTOR, 'ADCG'),
]

RESULTS_CSV = Path('/Users/ashrafshahreyar/Coding/Data Mining/FairGAD/reddit_results.csv')

torch.set_num_threads(12)

In [3]:
# ── Load dataset ───────────────────────────────────────────────────────────
data = torch.load(DATA_ROOT / 'reddit.pt')
data.y         = data.y.bool()
data.x         = data.x.float()
data.sensitive = data.sensitive.float()
contamination  = data.contamination

def sensitive_to_idx_dict(tensor):
    values, inv = np.unique(tensor.reshape(-1).cpu().numpy(), return_inverse=True)
    return {v: (inv == i).nonzero() for i, v in enumerate(values.tolist())}

sensitive_dict = sensitive_to_idx_dict(data.sensitive)
print(f'Nodes: {data.num_nodes} | Edges: {data.num_edges} | Anomalies: {data.y.sum().item()} | Contamination: {contamination:.4f}')

Nodes: 9892 | Edges: 1211748 | Anomalies: 1353 | Contamination: 0.1368


In [4]:
# ── Model factory ──────────────────────────────────────────────────────────
def make_model(name):
    kwargs = dict(batch_size=BATCH_SIZE, gpu=GPU, contamination=contamination, verbose=0)
    no_batch = dict(gpu=GPU, contamination=contamination, verbose=0)
    return {
        'DOMINANT': lambda: DOMINANT(**kwargs),
        'CONAD':    lambda: CONAD(**kwargs),
        'COLA':     lambda: CoLA(**kwargs),
        'ADONE':    lambda: AdONE(**kwargs),
        'DONE':     lambda: DONE(**kwargs),
        'ONE':      lambda: ONE_NEW(**no_batch),
        'VGOD':     lambda: VGOD(**no_batch),
    }[name]()

In [5]:
# ── Single experiment runner ───────────────────────────────────────────────
def run_experiment(model_name, regulariser, fair_factor, adcg_factor):
    seed_everything(SEED)
    aucs, sps, eos, auprcs, nan = [], [], [], [], 0

    for _ in tqdm.tqdm(range(NUM_TRIALS), desc=f'{model_name}/{regulariser}', leave=False):
        model = make_model(model_name)
        gc.collect(); torch.cuda.empty_cache()

        if fair_factor > 0 or adcg_factor > 0:
            model.fit_with_fairness(
                data,
                fair_factor=fair_factor,
                sens_var_col=data.sensitive,
                adcg_factor=adcg_factor,
                regulariser=regulariser,
            )
        else:
            model.fit(data)

        gc.collect(); torch.cuda.empty_cache()
        outlier_prob = model.predict_proba(data, method='unify')[:, 1]
        prediction   = model.predict(data)
        gc.collect(); torch.cuda.empty_cache()
        del model

        if np.any(np.isnan(outlier_prob)):
            nan += 1
            continue

        aucs.append(eval_roc_auc(data.y.numpy(), outlier_prob))
        sps.append(statistical_parity(prediction, sensitive_dict))
        eos.append(equality_of_odds(prediction, data.y.numpy(), sensitive_dict))
        p, r, _ = precision_recall_curve(data.y.numpy(), outlier_prob)
        auprcs.append(auc(r, p))

    return {
        'model':       model_name,
        'regulariser': regulariser,
        'fair_factor': fair_factor,
        'adcg_factor': adcg_factor,
        'dataset':     DATASET,
        'AUCROC_mean': np.mean(aucs),
        'AUCROC_std':  np.std(aucs),
        'SP_mean':     np.mean(sps),
        'SP_std':      np.std(sps),
        'EO_mean':     np.mean(eos),
        'EO_std':      np.std(eos),
        'AUPRC_mean':  np.mean(auprcs),
        'AUPRC_std':   np.std(auprcs),
        'nan_trials':  nan,
    }

In [6]:
# ── Run all experiments ────────────────────────────────────────────────────
# Total: 7 models × 5 conditions = 35 runs × 20 trials each
results = []

total = len(MODELS) * len(CONDITIONS)
done  = 0

for reg, ff, af, label in CONDITIONS:
    for model_name in MODELS:
        done += 1
        print(f'[{done}/{total}] {label} | {model_name}')
        row = run_experiment(model_name, reg, ff, af)
        results.append(row)

print('\nAll experiments complete.')

[1/35] Baseline | DOMINANT


[2/35] Baseline | CONAD


[3/35] Baseline | COLA


[4/35] Baseline | ADONE


[5/35] Baseline | DONE


...

[32/35] ADCG | ADONE


[33/35] ADCG | DONE


[34/35] ADCG | ONE


[35/35] ADCG | VGOD



All experiments complete.


In [7]:
# ── Build results DataFrame ────────────────────────────────────────────────
df = pd.DataFrame(results)

# Paper-style formatted columns
df['AUCROC'] = df.apply(lambda r: f"{r['AUCROC_mean']:.4f}±{r['AUCROC_std']:.4f}", axis=1)
df['SP']     = df.apply(lambda r: f"{r['SP_mean']:.4f}±{r['SP_std']:.4f}",     axis=1)
df['EO']     = df.apply(lambda r: f"{r['EO_mean']:.4f}±{r['EO_std']:.4f}",     axis=1)
df['AUPRC']  = df.apply(lambda r: f"{r['AUPRC_mean']:.4f}±{r['AUPRC_std']:.4f}", axis=1)

display_cols = ['model', 'regulariser', 'fair_factor', 'adcg_factor', 'AUCROC', 'SP', 'EO', 'AUPRC', 'nan_trials']
df[display_cols]

,model,regulariser,fair_factor,adcg_factor,AUCROC,SP,EO,AUPRC,nan_trials
0,DOMINANT,none,0.0000,0,0.6081±0.0008,0.1323±0.0014,0.0558±0.0019,0.2003±0.0006,0
1,CONAD,none,0.0000,0,0.6082±0.0009,0.1325±0.0011,0.0562±0.0020,0.2003±0.0006,0
2,COLA,none,0.0000,0,0.4497±0.0139,0.0534±0.0182,0.0354±0.0240,0.1847±0.0141,0
3,ADONE,none,0.0000,0,0.5542±0.0106,0.0783±0.0126,0.0192±0.0082,0.1894±0.0071,0
4,DONE,none,0.0000,0,0.5524±0.0078,0.0731±0.0074,0.0146±0.0070,0.1873±0.0045,0
5,ONE,none,0.0000,0,0.4963±0.0067,0.0088±0.0067,0.0139±0.0088,0.1354±0.0032,0
6,VGOD,none,0.0000,0,0.7205±0.0091,0.4267±0.0597,0.4715±0.0642,0.3934±0.0245,0
7,DOMINANT,hin,0.0001,0,0.6081±0.0008,0.1323±0.0014,0.0558±0.0019,0.2003±0.0006,0
8,CONAD,hin,0.0001,0,0.6082±0.0009,0.1325±0.0011,0.0562±0.0020,0.2003±0.0006,0
9,COLA,hin,0.0001,0,0.4497±0.0139,0.0534±0.0182,0.0354±0.0240,0.1847±0.0141,0


In [8]:
# ── Export ─────────────────────────────────────────────────────────────────
df.to_csv(RESULTS_CSV, index=False)
print(f'Results saved to {RESULTS_CSV}')

# Also print in the same pipe-delimited format as the original script
print('\nPipe-delimited (regulariser|fair_factor|adcg_factor|dataset|model|AUCROC|SP|EO|AUPRC|nan):')
for _, r in df.iterrows():
    print(f"{r['regulariser']}|{r['fair_factor']}|{r['adcg_factor']}|{r['dataset']}|{r['model']}|"
          f"{r['AUCROC']}|{r['SP']}|{r['EO']}|{r['AUPRC']}|{int(r['nan_trials'])}")

Results saved to /Users/ashrafshahreyar/Coding/Data Mining/FairGAD/reddit_results.csv

Pipe-delimited (regulariser|fair_factor|adcg_factor|dataset|model|AUCROC|SP|EO|AUPRC|nan):
none|0.0|0|our_reddit|DOMINANT|0.6081±0.0008|0.1323±0.0014|0.0558±0.0019|0.2003±0.0006|0
none|0.0|0|our_reddit|CONAD|0.6082±0.0009|0.1325±0.0011|0.0562±0.0020|0.2003±0.0006|0
none|0.0|0|our_reddit|COLA|0.4497±0.0139|0.0534±0.0182|0.0354±0.0240|0.1847±0.0141|0
none|0.0|0|our_reddit|ADONE|0.5542±0.0106|0.0783±0.0126|0.0192±0.0082|0.1894±0.0071|0
none|0.0|0|our_reddit|DONE|0.5524±0.0078|0.0731±0.0074|0.0146±0.0070|0.1873±0.0045|0
none|0.0|0|our_reddit|ONE|0.4963±0.0067|0.0088±0.0067|0.0139±0.0088|0.1354±0.0032|0
none|0.0|0|our_reddit|VGOD|0.7205±0.0091|0.4267±0.0597|0.4715±0.0642|0.3934±0.0245|0
hin|0.0001|0|our_reddit|DOMINANT|0.6081±0.0008|0.1323±0.0014|0.0558±0.0019|0.2003±0.0006|0
hin|0.0001|0|our_reddit|CONAD|0.6082±0.0009|0.1325±0.0011|0.0562±0.0020|0.2003±0.0006|0
hin|0.0001|0|our_reddit|COLA|0.4497±0.0139|